## Dominant POI Interaction Analysis and Significant POI Classification System

 ## System Overview
 This notebook implements a comprehensive trajectory analysis system that:
 1. Identifies dominant POI interactions using weighted metrics (80% dwell time + 20% inverse distance)
 2. Classifies significant POIs based on cross-trajectory patterns


In [ ]:
# ## 1. Environment Setup and Imports

# %%

# Data manipulation
import numpy as np
from helpers.fuseki_config import query_inference_endpoint, update_endpoint

# Semantic Web libraries
# Visualization
from helpers.fuseki_queries import execute_sparql_query, execute_sparql_update, format_results, load_query

In [ ]:
DWELL_WEIGHT = 0.8
DISTANCE_WEIGHT = 0.2
MIN_INTERACTIONS_THRESHOLD = 10
MIN_DOMINANT_THRESHOLD = 3


def calculate_interaction_score(dwell_seconds: str, distance: float) -> float:
    """
    Calculate weighted interaction score.
    Score = 0.8 * normalized_dwell_time + 0.2 * inverse_normalized_distance
    """

    # Normalize dwell time (using log scale for better distribution)
    # Adding 1 to avoid log(0)
    normalized_dwell = np.log1p(float(dwell_seconds)) / np.log1p(3600)  # Normalize to 2 hour max

    # Inverse distance normalization (closer = higher score)
    # Adding 1 to avoid division by zero
    normalized_distance = 1 / (1 + float(distance)/10)  # 10 meters as reference distance

    # Calculate weighted score
    score = (DWELL_WEIGHT * normalized_dwell +
            DISTANCE_WEIGHT * normalized_distance)

    return score

In [ ]:
meaningful_interactions_query = load_query("get-meaningful-interactions.sparql")
meaningful_interactions = execute_sparql_query(query_inference_endpoint, meaningful_interactions_query)
interactions_df = format_results(meaningful_interactions)

for index, interaction in interactions_df.iterrows():
    score = calculate_interaction_score(interaction["seconds"], interaction["distance"])
    interactions_df.at[index, 'score'] = score

    print(score, interaction["seconds"], interaction["distance"], interaction["interaction"])

In [ ]:
# Clear any existing DominantInteraction class assertions, then pick exactly one
# (top-scored, earliest-position on tie) per trajectory and assert it. This keeps
# NB 5 idempotent across re-runs; SHACL ex:DominantSingletonShape validates the invariant.
#
# Transport/navigation/utility POIs (BusStation, TrainStation, ParkingArea, ...) are
# excluded from dominant candidacy — they score high via dwell time (waiting at a stop)
# but carry no semantic signal for the pipeline's purpose/engagement tags.
execute_sparql_update(update_endpoint, """
PREFIX poi: <http://w3id.org/trajecto/poi#>
DELETE { ?int a poi:DominantInteraction }
WHERE  { ?int a poi:DominantInteraction }
""")

EXCLUDED_CATEGORIES = {
    'ATM', 'Bank', 'PostOffice',
    'BusStation', 'TrainStation', 'Airport', 'Transportation', 'TransportationPOI',
    'TransitStop', 'ParkingArea',
    'Elevator', 'Escalator', 'Staircase',
    'Entrance', 'Exit', 'Intersection', 'Junction',
    'Transition', 'TransitionPOI', 'Waypoint', 'NavigationLandmark',
    'Restroom', 'RestRoom', 'ServicePOI',
    'PointofInterest', 'PointOfInterest', 'SignificantPOI',
}

# Fetch every (interaction, POI-class) pair for meaningful interactions. A POI can
# have multiple rdf:type assertions due to subclass inference, so we exclude the
# interaction if ANY of its POI's classes fall in EXCLUDED_CATEGORIES.
categories_query = """
PREFIX poi: <http://w3id.org/trajecto/poi#>
PREFIX trajecto: <http://w3id.org/trajecto#>
SELECT ?interaction ?cat WHERE {
    ?interaction a poi:MeaningfulInteraction ;
                 poi:interactsWith ?p .
    ?p a ?cat .
    FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
}
"""
cat_df = format_results(execute_sparql_query(query_inference_endpoint, categories_query))
cat_df['cat_short'] = cat_df['cat'].str.split('#').str[-1].str.split('/').str[-1]
excluded_interactions = set(cat_df.loc[cat_df['cat_short'].isin(EXCLUDED_CATEGORIES), 'interaction'])

candidates = interactions_df[~interactions_df['interaction'].isin(excluded_interactions)]

# Break ties deterministically: highest score, then lowest positionIndex if available,
# else lexicographic on the interaction URI.
if 'positionIndex' in candidates.columns:
    candidates = candidates.sort_values(
        ['trajectory', 'score', 'positionIndex', 'interaction'],
        ascending=[True, False, True, True]
    )
else:
    candidates = candidates.sort_values(
        ['trajectory', 'score', 'interaction'],
        ascending=[True, False, True]
    )

dominant_interactions = candidates.drop_duplicates('trajectory', keep='first')

for _index, row in dominant_interactions.iterrows():
    iri = row['interaction']
    query = f"""
    PREFIX poi: <http://w3id.org/trajecto/poi#>
    INSERT DATA {{
        <{iri}> a poi:DominantInteraction ;
    }}
    """
    execute_sparql_update(update_endpoint, query)

n_total_traj = interactions_df['trajectory'].nunique()
n_excluded_ints = len(interactions_df) - len(candidates)
n_no_dom = n_total_traj - len(dominant_interactions)
print(f"Excluded {n_excluded_ints} transport/utility interactions from dominant candidacy.")
print(f"Asserted {len(dominant_interactions)} DominantInteractions (one per trajectory).")
print(f"{n_no_dom} of {n_total_traj} trajectories left without dominant "
      f"(all candidates were transport/utility, or zero meaningful interactions).")


In [ ]:
add_significant_interactions = load_query("significant-poi-interactions-count.sparql")
significant_interactions = execute_sparql_update(update_endpoint, add_significant_interactions)

In [ ]:
# Validate KG state against stage-05-poi-scoring SHACL shapes
from helpers.shacl_validator import validate_before_import, construct_validation_graph

proposed = construct_validation_graph('stage-05-poi-scoring')
print(f"Fetched {len(proposed)} triples for validation")
report = validate_before_import(proposed, 'stage-05-poi-scoring', halt_on_violations=True)